GPU Enabled

In [1]:
!nvidia-smi

Mon Aug 17 12:54:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Full-corpus run: gemma4:12b, Bavaria only

Scope: just Bavaria (`by`, AfD entry 2018-11-05), not the full ±1yr window across all
states -- 72,089 classifiable paragraphs vs. 1,144,158. Reason: Colab Pro has a **hard 24h
session cap** regardless of activity, and the full corpus would take an estimated ~56 days
at gemma4's local Mac rate -- Bavaria alone is ~3.5 days, i.e. only a handful of reconnects
instead of ~55.

`score_with_model.py --full-corpus` checkpoints every row to disk as it's scored and resumes
automatically from wherever it left off -- so when this session dies at the 24h mark, just
reconnect and re-run the last cell below. Uses the v2 window+flag prompt (see
`impoliteness_lib.py`'s `PROMPT_VERSION`).

Uses `gemma4:12b` -- NOT `gemma4:e2b` (that's the small edge/on-device variant, not the model
used in the interrater comparison).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DATA_ROOT'] = "/content/drive/MyDrive/my_projects/M.A. Parliament/Code and Data/data"

Mounted at /content/drive


In [3]:
# Repo is public -- plain clone, no auth needed.
import os

if not os.path.isdir('/content/Incivility-in-Plenary-de'):
    !git clone https://github.com/AnLeWe/Incivility-in-Plenary-de.git /content/Incivility-in-Plenary-de
else:
    !cd /content/Incivility-in-Plenary-de && git pull

%cd /content/Incivility-in-Plenary-de
!pip install -q -r requirements.txt

Cloning into '/content/Incivility-in-Plenary-de'...
remote: Enumerating objects: 326, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 326 (delta 152), reused 314 (delta 140), pack-reused 0 (from 0)
Receiving objects: 100% (326/326), 1.46 MiB | 3.70 MiB/s, done.
Resolving deltas: 100% (152/152), done.
/content/Incivility-in-Plenary-de
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.6/349.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 157.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 121.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are i

Pull the model -- run in the terminal (Runtime → "Open terminal" or the `!` cell below), not
inline, same reason as before (progress bar doesn't render well in a notebook cell):

```
ollama pull gemma4:12b
```

In [4]:
!sudo apt-get update -y
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!OLLAMA_NUM_PARALLEL=8 nohup ollama serve > /content/ollama_serve.log 2>&1 &
!sleep 5 && ollama pull gemma4:12b
!ollama list


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [107 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,908 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,164 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
G

## Concurrency benchmark (this GPU specifically)

`OLLAMA_NUM_PARALLEL=8` is set inline in the install/serve cell above -- if you already ran
that cell once before this was added (check: does its output say `OLLAMA_NUM_PARALLEL:1` or
`:8`?), restart the Colab runtime (Runtime → Restart session) once and re-run from the top,
since an already-running server won't pick up a changed env var on its own.

Times real Bavaria rows (not toy prompts) at concurrency 1/2/4/8/16 with `gemma4:12b`, same
method as the earlier Mac benchmark. That one plateaued at concurrency=2 with only ~20%
gain -- compute-bound on that GPU. This A100 has a very different compute/VRAM ratio, so
don't assume the same ceiling; measure it here instead. Pick whatever concurrency actually
helps and set it in the `CONCURRENCY` variable in the next cell.

```python
import sys, time
sys.path.insert(0, '/content/Incivility-in-Plenary-de/measurement')
from concurrent.futures import ThreadPoolExecutor

import ollama
from impoliteness_lib import build_classification_pool, build_prompt, PROMPT_VERSION

pool = build_classification_pool(os.environ['DATA_ROOT'], verbose=False)
sample_texts = (
    pool.dedup[pool.dedup['state'] == 'by']['text_to_classify']
    .dropna().astype(str).head(16).tolist()
)

def call_one(text):
    t0 = time.time()
    ollama.chat(
        model='gemma4:12b',
        messages=build_prompt(text, prompt_version=PROMPT_VERSION),
        format='json', think=False,
        options={'temperature': 0, 'seed': 20260723, 'num_ctx': 40960},
    )
    return time.time() - t0

for concurrency in (1, 2, 4, 8, 16):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        list(ex.map(call_one, sample_texts))
    wall = time.time() - t0
    print(f"concurrency={concurrency}: {len(sample_texts)} items in {wall:.1f}s "
          f"({wall/len(sample_texts):.2f} sec/item wall-clock, "
          f"{len(sample_texts)/wall:.2f} items/sec throughput)")
```

In [5]:
# Set this from the benchmark above -- whichever concurrency actually gave a real
# speedup (not just the highest number tried). 1 = sequential, same as before.
CONCURRENCY = 4

## Four separate variant runs, one cell each

Same idea as the local gemma4 ablation (n=2000 sample), but here on the full Bavaria corpus
(72,089 rows) instead of the fixed sample -- `baseline` is the actual v1 prompt (matches the
already-run n=2000 four-model comparison), `window`/`flag` isolate each half of v2's context,
`window_flag` is the full v2 design. Each is its own cell/output file so you can watch each
one separately rather than one chained loop.

Still run the four **cells** one after another, not simultaneously in separate tabs -- one
model instance, one GPU, running two cells at once would just contend for it. Within a single
cell, `--concurrency {CONCURRENCY}` (set from the benchmark above) sends that many requests to
Ollama at once, which is a different thing and is what this benchmark was for. Each cell is
independently checkpointed/resumable across 24h reconnects.

Rough time budget: ~3.5 days/variant, ~14 days for all four back to back, at `--concurrency 1`
and based on the local Mac gemma4 rate (~4.2s/item) -- both of those should be pessimistic
here: this session runs on an A100-SXM4-80GB (`nvidia-smi` above), not a T4, and the benchmark
above may justify a higher CONCURRENCY. Watch each cell's progress printout (every 25 items)
for the actual rate.

In [6]:
# baseline -- actual v1 prompt, no context/flag at all (matches the n=2000 four-model comparison)
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant baseline --state by --platform-label colab_a100 --concurrency {CONCURRENCY}

In [ ]:
# window only -- +-1 paragraph context, ordnungsruf_follows suppressed
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant window --state by --platform-label colab_a100 --concurrency {CONCURRENCY}

here using T4 for now:

In [ ]:
# flag only -- ordnungsruf_follows hint, no +-1 window
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant flag --state by --platform-label colab_t4 --concurrency {CONCURRENCY}

In [ ]:
# Run this cell to start, and re-run it (after reconnecting) whenever the session dies --
# it resumes from wherever impoliteness_full_by_gemma4-12b_window_flag.csv already has rows.
%cd /content/Incivility-in-Plenary-de/measurement
!python score_with_model.py --model gemma4:12b --variant window_flag --state by --platform-label colab_t4 --concurrency {CONCURRENCY}

/content/Incivility-in-Plenary-de/measurement
Building classification pool (scope=by)...


## With Transformers

Local Inference on GPU
Model page: https://huggingface.co/google/gemma-4-12B-it

⚠️ If the generated code snippets do not work, please open an issue on either the model repo and/or on huggingface.js 

In [ ]:
!pip install -U transformers

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("google/gemma-4-12B-it")
model = AutoModelForMultimodalLM.from_pretrained("google/gemma-4-12B-it", device_map="auto")